# 🎯 Fitts' Law Analysis - ALL 12 PARTICIPANTS

This notebook analyzes data from ALL 12 students by:
- Using pre-calculated results where available
- Calculating from raw data where results are missing

## 🚀 How to Use:
1. Run Cell 1 (Setup)
2. Run Cell 2 (Upload) - Upload fitts-student-data.zip
3. Run remaining cells to see results from all 12 participants!

In [ ]:
# 1️⃣ Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
import os
import glob
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
print("✅ Packages loaded!")

In [ ]:
# 2️⃣ Upload Data
from google.colab import files
import zipfile

print("📁 Upload your fitts-student-data.zip file:")
uploaded = files.upload()

for filename in uploaded.keys():
    if filename.endswith('.zip'):
        print(f"\n📦 Extracting {filename}...")
        with zipfile.ZipFile(filename, 'r') as zip_ref:
            zip_ref.extractall('.')
        print(f"✅ Extracted!")

In [ ]:
# 3️⃣ Load ALL Data - Raw and Results
print("🔍 Loading data from ALL 12 participants...")
print("="*80)

student_folders = [d for d in os.listdir('fitts-student-data')
                   if os.path.isdir(os.path.join('fitts-student-data', d))]

print(f"\nFound {len(student_folders)} student folders\n")

# Load raw data
raw_data_by_student = {}
for student in student_folders:
    student_path = os.path.join('fitts-student-data', student)
    raw_files = glob.glob(os.path.join(student_path, '*raw-data*.csv'))
    
    if raw_files:
        df = pd.read_csv(raw_files[0])
        df['ParticipantID'] = student
        raw_data_by_student[student] = df
        print(f"   ✅ {student}: {len(df)} trials (raw data)")

# Load pre-calculated results
results_by_student = {}
for student in student_folders:
    student_path = os.path.join('fitts-student-data', student)
    results_files = glob.glob(os.path.join(student_path, '*results*.csv'))
    
    if results_files:
        df = pd.read_csv(results_files[0])
        df['ParticipantID'] = student
        results_by_student[student] = df
        print(f"   ✅ {student}: {len(df)} conditions (pre-calculated results)")

print(f"\n📊 Summary:")
print(f"   Students with raw data: {len(raw_data_by_student)}")
print(f"   Students with results: {len(results_by_student)}")

In [ ]:
# 4️⃣ Calculate Results for Students Missing Pre-calculated Results
print("\n" + "="*80)
print("📊 CALCULATING FITTS METRICS FOR ALL 12 PARTICIPANTS")
print("="*80)

all_results = []

# For each student
for student in student_folders:
    print(f"\n🔄 Processing {student}...")
    
    # Check if we have pre-calculated results
    if student in results_by_student:
        print(f"   ✅ Using pre-calculated results ({len(results_by_student[student])} conditions)")
        all_results.append(results_by_student[student])
    
    # Otherwise, calculate from raw data
    elif student in raw_data_by_student:
        print(f"   🔧 Calculating from raw data ({len(raw_data_by_student[student])} trials)...")
        
        raw_df = raw_data_by_student[student]
        student_results = []
        
        # Group by filter and layout conditions
        grouping_cols = ['FilterType', 'TargetSize', 'Amplitude']
        grouping_cols = [col for col in grouping_cols if col in raw_df.columns]
        
        for group_keys, group in raw_df.groupby(grouping_cols):
            # Calculate Fitts metrics
            Ae = group['ActualAmplitude'].mean()
            
            # ISO 9241-9 directional projection: project endpoint deviation onto movement direction
            if 'Direction' in group.columns and 'TargetX' in group.columns:
                theta_rad = np.radians(group['Direction'].astype(float))
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                projections = dx * np.cos(theta_rad) + dy * np.sin(theta_rad)
                We = 4.133 * projections.std()
            else:
                dx = group['SelectionX'] - group['TargetX']
                dy = group['SelectionY'] - group['TargetY']
                distances = np.sqrt(dx**2 + dy**2)
                We = 4.133 * distances.std()
            
            IDe = np.log2(Ae / We + 1) if We > 0 else 0
            MeanMT = group['MovementTime'].mean()
            TP = IDe / MeanMT if MeanMT > 0 else 0
            
            result = {
                'ParticipantID': student,
                'FilterType': group_keys[0] if len(grouping_cols) > 0 else None,
                'TargetSize': group_keys[1] if len(grouping_cols) > 1 else None,
                'Amplitude': group_keys[2] if len(grouping_cols) > 2 else None,
                'N': len(group),
                'MeanMT': MeanMT,
                'Ae': Ae,
                'We': We,
                'IDe': IDe,
                'TP': TP
            }
            student_results.append(result)
        
        student_df = pd.DataFrame(student_results)
        all_results.append(student_df)
        print(f"   ✅ Calculated {len(student_df)} conditions")
    
    else:
        print(f"   ⚠️  No data found for {student}")

# Combine all results
results_data = pd.concat(all_results, ignore_index=True)

print(f"\n" + "="*80)
print(f"✅ COMPLETE! Analyzed {results_data['ParticipantID'].nunique()} participants")
print(f"   Total conditions: {len(results_data)}")
print(f"   Participants: {sorted(results_data['ParticipantID'].unique())}")
print("="*80)

# Show preview
print("\n📊 Data preview:")
display(results_data.head(10))

In [ ]:
# 5️⃣ Summary Statistics - ALL 12 PARTICIPANTS
print("="*80)
print("📊 SUMMARY STATISTICS - ALL 12 PARTICIPANTS")
print("="*80)

print(f"\n👥 Total Participants: {results_data['ParticipantID'].nunique()}")
print(f"📊 Total Conditions Analyzed: {len(results_data)}")

if 'FilterType' in results_data.columns:
    print(f"\n🎯 Overall Performance:")
    print(f"   Mean Throughput: {results_data['TP'].mean():.3f} ± {results_data['TP'].std():.3f} bits/s")
    print(f"   Mean Movement Time: {results_data['MeanMT'].mean():.3f} ± {results_data['MeanMT'].std():.3f} s")
    
    print(f"\n🔧 Performance by Filter Type:")
    filter_stats = results_data.groupby('FilterType').agg({
        'TP': ['mean', 'std', 'min', 'max', 'count'],
        'MeanMT': ['mean', 'std']
    }).round(3)
    display(filter_stats)
    
    print(f"\n👥 Participants per Filter:")
    for ftype in results_data['FilterType'].unique():
        n_participants = results_data[results_data['FilterType'] == ftype]['ParticipantID'].nunique()
        print(f"   {ftype}: {n_participants} participants")

In [ ]:
# 6️⃣ Statistical Tests - ALL 12 PARTICIPANTS
print("="*80)
print("📊 STATISTICAL TESTS - ALL 12 PARTICIPANTS")
print("="*80)

if 'FilterType' in results_data.columns:
    filters = results_data['FilterType'].unique()
    
    if len(filters) == 2:
        f1, f2 = filters
        
        # Get throughput by participant
        tp1 = results_data[results_data['FilterType'] == f1].groupby('ParticipantID')['TP'].mean()
        tp2 = results_data[results_data['FilterType'] == f2].groupby('ParticipantID')['TP'].mean()
        
        print(f"\n📈 Throughput Comparison:")
        print(f"   {f1}: {tp1.mean():.3f} ± {tp1.std():.3f} bits/s (N={len(tp1)} participants)")
        print(f"   {f2}: {tp2.mean():.3f} ± {tp2.std():.3f} bits/s (N={len(tp2)} participants)")
        
        # Find common participants (for paired test)
        common = set(tp1.index) & set(tp2.index)
        print(f"\n   Participants with both filters: {len(common)}")
        
        if len(common) > 1:
            tp1_vals = [tp1[p] for p in common]
            tp2_vals = [tp2[p] for p in common]
            
            # Paired t-test
            t_stat, p_val = stats.ttest_rel(tp1_vals, tp2_vals)
            
            print(f"\n   Paired t-test (N={len(common)}): t({len(common)-1})={t_stat:.3f}, p={p_val:.4f}")
            
            if p_val < 0.05:
                better = f1 if np.mean(tp1_vals) > np.mean(tp2_vals) else f2
                worse = f2 if better == f1 else f1
                improvement = abs((np.mean(tp1_vals) - np.mean(tp2_vals)) / min(np.mean(tp1_vals), np.mean(tp2_vals)) * 100)
                print(f"   ✅ SIGNIFICANT DIFFERENCE (p < 0.05)")
                print(f"   🏆 {better} is {improvement:.1f}% better than {worse}")
            else:
                print(f"   ❌ No significant difference (p >= 0.05)")
            
            # Cohen's d
            diff = np.array(tp1_vals) - np.array(tp2_vals)
            cohens_d = np.mean(diff) / np.std(diff, ddof=1)
            print(f"   Effect size (Cohen's d): {abs(cohens_d):.3f}")
            
            if abs(cohens_d) < 0.2:
                print(f"      → Small effect")
            elif abs(cohens_d) < 0.5:
                print(f"      → Medium effect")
            else:
                print(f"      → Large effect")
        
        # Also do independent t-test for all data
        print(f"\n   Independent t-test (all participants):")
        all_tp1 = results_data[results_data['FilterType'] == f1]['TP']
        all_tp2 = results_data[results_data['FilterType'] == f2]['TP']
        t_stat_ind, p_val_ind = stats.ttest_ind(all_tp1, all_tp2)
        print(f"   t={t_stat_ind:.3f}, p={p_val_ind:.4f}")
        if p_val_ind < 0.05:
            print(f"   ✅ Significant (p < 0.05)")
        else:
            print(f"   ❌ Not significant (p >= 0.05)")

In [ ]:
# 7️⃣ Graph 1: Throughput by Filter - ALL 12
if 'FilterType' in results_data.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=results_data, x='FilterType', y='TP', palette='Set2')
    sns.swarmplot(data=results_data, x='FilterType', y='TP', color='black', alpha=0.3, size=3)
    plt.title('Throughput by Filter Type (N=12 participants)', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Filter Type', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('throughput_by_filter_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: throughput_by_filter_N12.png")

In [ ]:
# 8️⃣ Graph 2: Movement Time by Filter - ALL 12
if 'FilterType' in results_data.columns:
    plt.figure(figsize=(10, 6))
    sns.boxplot(data=results_data, x='FilterType', y='MeanMT', palette='Set2')
    sns.swarmplot(data=results_data, x='FilterType', y='MeanMT', color='black', alpha=0.3, size=3)
    plt.title('Movement Time by Filter Type (N=12 participants)', fontsize=16, fontweight='bold')
    plt.ylabel('Movement Time (s)', fontsize=12)
    plt.xlabel('Filter Type', fontsize=12)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('movement_time_by_filter_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: movement_time_by_filter_N12.png")

In [ ]:
# 9️⃣ Graph 3: Individual Participants - ALL 12
if 'FilterType' in results_data.columns:
    participant_tp = results_data.groupby(['ParticipantID', 'FilterType'])['TP'].mean().reset_index()
    
    plt.figure(figsize=(16, 6))
    sns.barplot(data=participant_tp, x='ParticipantID', y='TP', hue='FilterType', palette='Set2')
    plt.title('Individual Participant Throughput (N=12)', fontsize=16, fontweight='bold')
    plt.ylabel('Throughput (bits/s)', fontsize=12)
    plt.xlabel('Participant', fontsize=12)
    plt.xticks(rotation=45, ha='right')
    plt.legend(title='Filter')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig('participant_throughput_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: participant_throughput_N12.png")

In [ ]:
# 🔟 Graph 4: Fitts' Law Regression - ALL 12
if 'IDe' in results_data.columns and 'FilterType' in results_data.columns:
    plt.figure(figsize=(10, 6))
    
    for ftype in results_data['FilterType'].unique():
        data = results_data[results_data['FilterType'] == ftype]
        plt.scatter(data['IDe'], data['MeanMT'], label=ftype, alpha=0.6, s=50)
        
        z = np.polyfit(data['IDe'], data['MeanMT'], 1)
        p = np.poly1d(z)
        x_line = np.linspace(data['IDe'].min(), data['IDe'].max(), 100)
        plt.plot(x_line, p(x_line), '--', linewidth=2)
    
    plt.xlabel('Index of Difficulty (bits)', fontsize=12)
    plt.ylabel('Movement Time (s)', fontsize=12)
    plt.title("Fitts' Law: MT = a + b × ID (N=12)", fontsize=16, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('fitts_law_regression_N12.png', dpi=300, bbox_inches='tight')
    plt.show()
    print("💾 Saved: fitts_law_regression_N12.png")

In [ ]:
# 1️⃣1️⃣ Export Results - ALL 12
if 'FilterType' in results_data.columns:
    summary_filter = results_data.groupby('FilterType').agg({
        'TP': ['mean', 'std', 'min', 'max'],
        'MeanMT': ['mean', 'std', 'min', 'max']
    }).round(4)
    summary_filter.to_csv('summary_by_filter_N12.csv')
    print("💾 Saved: summary_by_filter_N12.csv")

summary_participant = results_data.groupby(['ParticipantID', 'FilterType']).agg({
    'TP': 'mean',
    'MeanMT': 'mean'
}).round(4)
summary_participant.to_csv('summary_by_participant_N12.csv')
print("💾 Saved: summary_by_participant_N12.csv")

results_data.to_csv('all_results_N12.csv', index=False)
print("💾 Saved: all_results_N12.csv")

print("\n✅ COMPLETE! All 12 participants analyzed.")
print("\n📊 Files generated:")
print("   • 4 graphs (PNG, 300 DPI)")
print("   • 3 CSV summary tables")
print("   • Statistical test results above")